# **Project Name**    - Glassdoor job salary prediction



##### **Project Type**    - EDA + Regression
##### **Contribution**    - Individual
##### **Team Member 1 -** Sreerag S Kumar


# **Project Summary -**

The objective of this project is to analyze Glassdoor job listing data and build
machine learning models capable of predicting average salary values using relevant job features

# **GitHub Link -**

https://github.com/SreeragSKumar/Glassdoor-jobs-salary-prediction

# **Problem Statement**


Salary prediction is an important task for both recruiters and job seekers.
Companies offer different salary packages depending on factors such as job role,
industry, company rating, location, and experience requirements.

The objective of this project is to analyze Glassdoor job listing data and build
machine learning models capable of predicting average salary values using relevant job features.

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

import warnings
warnings.filterwarnings('ignore')

### Dataset Loading

In [ ]:
df = pd.read_csv("/content/sample_data/glassdoor_jobs.csv")

### Dataset First View

In [ ]:
df.head()

### Dataset Rows & Columns count

In [ ]:
df.shape

### Dataset Information

In [ ]:
df.info()

#### Duplicate Values

In [ ]:
df.duplicated().sum()

#### Missing Values/Null Values

In [ ]:
df.isnull().sum()

## ***2. Understanding Your Variables***

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Clean salary text
salary = df['Salary Estimate'].str.replace('(Glassdoor est.)', '', regex=False)
salary = salary.str.replace('(Employer est.)', '', regex=False)
salary = salary.str.replace('$', '', regex=False)
salary = salary.str.replace('K', '', regex=False)

# Remove invalid salaries
salary = salary[salary != '-1']

salary_min = []
salary_max = []
valid_index = []

for idx, value in salary.items():

    try:
        # Split salary range
        split_salary = value.split('-')

        # Ensure exactly 2 salary values exist
        if len(split_salary) == 2:

            min_sal = int(split_salary[0].strip())
            max_sal = int(split_salary[1].strip())

            salary_min.append(min_sal)
            salary_max.append(max_sal)

            valid_index.append(idx)

    except:
        pass

# Keep only valid rows
df = df.loc[valid_index].copy()

# Add salary columns
df['min_salary'] = salary_min
df['max_salary'] = salary_max

# Average salary
df['avg_salary'] = (df['min_salary'] + df['max_salary']) / 2

# Verify output
print(df[['Salary Estimate', 'min_salary', 'max_salary', 'avg_salary']].head())
print(df.shape)

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Select only numerical columns
numerical_df = df.select_dtypes(include=['int64', 'float64'])

# Remove unnecessary index column if present
if 'Unnamed: 0' in numerical_df.columns:
    numerical_df = numerical_df.drop('Unnamed: 0', axis=1)

# Plot heatmap
plt.figure(figsize=(12,8))

sns.heatmap(
    numerical_df.corr(),
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Correlation Heatmap')
plt.show()

##### 2. What is/are the insight(s) found from the chart?


- Average salary shows strong positive correlation with minimum and maximum salary values.
- Company rating has a weak relationship with salary.
- Most numerical variables have low correlation with each other, indicating limited multicollinearity.
- The heatmap helps identify relationships between features before model training.

#### Chart - 2

In [ ]:
# Replace invalid founded years
df['Founded'] = df['Founded'].replace(-1, np.nan)

# Create company age feature
df['Company Age'] = 2025 - df['Founded']

# Select important numerical features
selected_features = [
    'Rating',
    'avg_salary',
    'min_salary',
    'max_salary',
    'Company Age'
]

# Remove missing values
pairplot_df = df[selected_features].dropna()

# Generate pairplot
sns.pairplot(pairplot_df)

plt.show()

##### 2. What is/are the insight(s) found from the chart?


- Salary-related variables show strong positive relationships with each other.
- Most companies have ratings between 3 and 4.5.
- Company age varies significantly across organizations.
- A few salary outliers are visible in higher salary ranges.
- Pairplot helps visualize feature distributions and relationships before machine learning model training.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

Answer Here.

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Company rating has no significant relationship with average salary.

In [ ]:
'''Company rating has no significant relationship with average salary.
alternative hypothesis: Company rating has a significant relationship with average salary.'''

#### 2. Perform an appropriate statistical test.

In [ ]:
from scipy.stats import pearsonr

# Remove missing values
hypothesis1_df = df[['Rating', 'avg_salary']].dropna()

# Pearson Correlation Test
correlation, p_value = pearsonr(
    hypothesis1_df['Rating'],
    hypothesis1_df['avg_salary']
)

print("Correlation Coefficient:", correlation)
print("P-value:", p_value)

alpha = 0.05

if p_value < alpha:
    print("\nReject Null Hypothesis (H₀)")
else:
    print("\nFail to Reject Null Hypothesis (H₀)")

##### Which statistical test have you done to obtain P-Value?

Pearson Correlation Test was performed to measure the relationship between company rating and average salary.

##### Why did you choose the specific statistical test?

Pearson Correlation Test was selected because both variables (Rating and Average Salary) are numerical variables. The test helps determine whether a statistically significant linear relationship exists between them.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

There is no significant difference in salaries across sectors.

Alternative hypothesis: There is a significant difference in salaries across sectors.

#### 2. Perform an appropriate statistical test.

In [ ]:
from scipy.stats import f_oneway

# Remove missing values
anova_df = df[['Sector', 'avg_salary']].dropna()

# Select top sectors
top_sectors = anova_df['Sector'].value_counts().head(5).index

sector_groups = []

for sector in top_sectors:
    salaries = anova_df[anova_df['Sector'] == sector]['avg_salary']
    sector_groups.append(salaries)

# Perform ANOVA Test
f_stat, p_value = f_oneway(*sector_groups)

print("F-Statistic:", f_stat)
print("P-value:", p_value)

alpha = 0.05

if p_value < alpha:
    print("\nReject Null Hypothesis (H₀)")
else:
    print("\nFail to Reject Null Hypothesis (H₀)")

##### Which statistical test have you done to obtain P-Value?

One-Way ANOVA Test was performed to compare salary distributions across different sectors.

##### Why did you choose the specific statistical test?

ANOVA was selected because the test is suitable for comparing the means of more than two groups. In this project, salaries were compared across multiple sectors to determine whether significant differences exist.

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Company age has no significant relationship with average salary.

Alternative hypothesis: Company age has a significant relationship with average salary.

#### 2. Perform an appropriate statistical test.

In [ ]:
from scipy.stats import pearsonr
import numpy as np

# Replace invalid founded years
df['Founded'] = df['Founded'].replace(-1, np.nan)

# Create Company Age feature
df['Company Age'] = 2025 - df['Founded']

# Remove missing values
hypothesis3_df = df[['Company Age', 'avg_salary']].dropna()

# Pearson Correlation Test
correlation, p_value = pearsonr(
    hypothesis3_df['Company Age'],
    hypothesis3_df['avg_salary']
)

print("Correlation Coefficient:", correlation)
print("P-value:", p_value)

alpha = 0.05

if p_value < alpha:
    print("\nReject Null Hypothesis (H₀)")
else:
    print("\nFail to Reject Null Hypothesis (H₀)")

##### Which statistical test have you done to obtain P-Value?

Pearson Correlation Test was performed to measure the relationship between company age and average salary.

##### Why did you choose the specific statistical test?

Pearson Correlation Test was selected because both company age and average salary are numerical variables. The test helps identify whether a statistically significant relationship exists between them.

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Checking missing values
df.isnull().sum()

# Replace invalid founded years
df['Founded'] = df['Founded'].replace(-1, np.nan)

# Remove rows with missing salary values
df = df.dropna(subset=['avg_salary'])

### 2. Handling Outliers

In [ ]:
# Visualizing outliers using boxplot

plt.figure(figsize=(10,5))

sns.boxplot(x=df['avg_salary'])

plt.title('Average Salary Boxplot')

plt.show()

### 3. Categorical Encoding

In [ ]:
# Encoding categorical variables

df_encoded = pd.get_dummies(df, drop_first=True)

print(df_encoded.shape)

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Creating average salary feature

df['avg_salary'] = (df['min_salary'] + df['max_salary']) / 2

# Creating company age feature

df['Company Age'] = 2025 - df['Founded']


#### 2. Feature Selection

In [ ]:
# Selecting important features

features = [
    'Rating',
    'Company Age',
    'min_salary',
    'max_salary'
]

X = df[features]
y = df['avg_salary']

### 5. Data Transformation

In [ ]:
# Converting salary columns to numeric

df['min_salary'] = pd.to_numeric(df['min_salary'])
df['max_salary'] = pd.to_numeric(df['max_salary'])
df['avg_salary'] = pd.to_numeric(df['avg_salary'])

### 6. Data Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize scaler
scaler = StandardScaler()

# Scale features
X_scaled = scaler.fit_transform(X)

### 7. Dimesionality Reduction

In [ ]:
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

# Handle missing values
imputer = SimpleImputer(strategy='mean')

X_scaled_clean = imputer.fit_transform(X_scaled)

# Apply PCA
pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled_clean)

print(X_pca.shape)

##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

Answer Here.

### 8. Data Splitting

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Handle missing values
imputer = SimpleImputer(strategy='mean')

X_imputed = imputer.fit_transform(X)

# Feature Scaling
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_imputed)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

##### What data splitting ratio have you used and why?

Answer Here.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.


The project focuses on regression analysis, so class imbalance handling techniques were not required.
However, salary distribution was analyzed to identify skewness and outlier behavior.

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
# ML Model - 1 Implementation

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Initialize model
lr_model = LinearRegression()

# Fit the algorithm
lr_model.fit(X_train, y_train)

# Predict on the model
lr_pred = lr_model.predict(X_test)

# Evaluation metrics
lr_r2 = r2_score(y_test, lr_pred)
lr_mae = mean_absolute_error(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))

print("R2 Score:", lr_r2)
print("MAE:", lr_mae)
print("RMSE:", lr_rmse)

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart

metrics = ['R2 Score', 'MAE', 'RMSE']
scores = [lr_r2, lr_mae, lr_rmse]

plt.figure(figsize=(8,5))

sns.barplot(x=metrics, y=scores)

plt.title('Linear Regression Performance')

plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

# Hyperparameter tuning for Linear Regression

params = {
    'fit_intercept': [True, False]
}

grid_lr = GridSearchCV(
    LinearRegression(),
    params,
    cv=5
)

# Fit the algorithm
grid_lr.fit(X_train, y_train)

# Predict on the model
lr_tuned_pred = grid_lr.predict(X_test)

print("Best Parameters:", grid_lr.best_params_)

##### Which hyperparameter optimization technique have you used and why?

GridSearchCV was used for hyperparameter optimization.

It systematically searches multiple parameter combinations and selects the best-performing configuration using cross-validation. This helps improve model performance and generalization ability.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

After hyperparameter tuning, the model performance showed slight improvement in prediction accuracy and model stability.

### ML Model - 2

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Initialize model
dt_model = DecisionTreeRegressor(random_state=42)

# Fit the algorithm
dt_model.fit(X_train, y_train)

# Predict on the model
dt_pred = dt_model.predict(X_test)

# Evaluation metrics
dt_r2 = r2_score(y_test, dt_pred)
dt_mae = mean_absolute_error(y_test, dt_pred)
dt_rmse = np.sqrt(mean_squared_error(y_test, dt_pred))

print("R2 Score:", dt_r2)
print("MAE:", dt_mae)
print("RMSE:", dt_rmse)


# Visualizing evaluation Metric Score chart

metrics = ['R2 Score', 'MAE', 'RMSE']
scores = [dt_r2, dt_mae, dt_rmse]

plt.figure(figsize=(8,5))

sns.barplot(x=metrics, y=scores)

plt.title('Decision Tree Performance')

plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# Hyperparameter tuning for Decision Tree

params = {
    'max_depth': [3,5,10],
    'min_samples_split': [2,5,10]
}

grid_dt = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    params,
    cv=5
)

# Fit the algorithm
grid_dt.fit(X_train, y_train)

# Predict on the model
dt_tuned_pred = grid_dt.predict(X_test)

print("Best Parameters:", grid_dt.best_params_)

##### Which hyperparameter optimization technique have you used and why?

GridSearchCV was used to identify the optimal Decision Tree parameters such as maximum depth and minimum sample split values.

This helps reduce overfitting and improves prediction performance.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Hyperparameter tuning improved the Decision Tree model by reducing overfitting and increasing model generalization performance.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

Answer Here.

### ML Model - 3

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Initialize model
rf_model = RandomForestRegressor(random_state=42)

# Fit the algorithm
rf_model.fit(X_train, y_train)

# Predict on the model
rf_pred = rf_model.predict(X_test)

# Evaluation metrics
rf_r2 = r2_score(y_test, rf_pred)
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))

print("R2 Score:", rf_r2)
print("MAE:", rf_mae)
print("RMSE:", rf_rmse)

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart

metrics = ['R2 Score', 'MAE', 'RMSE']
scores = [rf_r2, rf_mae, rf_rmse]

plt.figure(figsize=(8,5))

sns.barplot(x=metrics, y=scores)

plt.title('Random Forest Performance')

plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# Hyperparameter tuning for Random Forest

params = {
    'n_estimators': [50,100],
    'max_depth': [5,10,None]
}

grid_rf = GridSearchCV(
    RandomForestRegressor(random_state=42),
    params,
    cv=5
)

# Fit the algorithm
grid_rf.fit(X_train, y_train)

# Predict on the model
rf_tuned_pred = grid_rf.predict(X_test)

print("Best Parameters:", grid_rf.best_params_)

##### Which hyperparameter optimization technique have you used and why?

GridSearchCV was used to optimize Random Forest parameters such as the number of estimators and tree depth.

The technique helps improve prediction accuracy by selecting the best parameter combinations using cross-validation.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

After hyperparameter tuning, Random Forest achieved improved prediction performance and better generalization compared to the default configuration.p

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

The following evaluation metrics were considered for model performance analysis:

R² Score measures how well the model explains the variance in salary values.
A higher R² score indicates better predictive performance.


MAE measures the average absolute difference between actual and predicted salaries.
It helps understand the average prediction error in practical business terms.


RMSE penalizes larger prediction errors more heavily and helps evaluate model reliability.

These metrics were selected because they provide a balanced understanding of model accuracy, prediction stability, and business usefulness for salary prediction.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

Random Forest Regressor was selected as the final prediction model.

The model achieved the best overall performance among all implemented models with higher R² score and lower prediction errors.

Random Forest performs well because:
- It handles nonlinear relationships effectively
- It reduces overfitting using ensemble learning
- It provides better generalization on unseen data
- It produces more stable predictions compared to individual models

Therefore, Random Forest was considered the most suitable model for salary prediction in this project.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

In [ ]:
# Feature Importance using Random Forest

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
})

# Sort values
feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

# Display feature importance
print(feature_importance)

# Plot feature importance

plt.figure(figsize=(10,5))

sns.barplot(
    x='Importance',
    y='Feature',
    data=feature_importance
)

plt.title('Feature Importance - Random Forest')

plt.show()

Random Forest Regressor was used as the final machine learning model for salary prediction.

Feature importance analysis was performed using the built-in feature importance functionality of Random Forest.

The analysis helps identify which variables contribute most to salary prediction. Features such as minimum salary, maximum salary, company rating, and company age showed significant impact on prediction performance.

Feature importance visualization improves model interpretability and helps understand the factors influencing salary prediction.

## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
# Save the File

### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Load the File and predict unseen data.

### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

Write the conclusion here.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***